In [7]:
#%pip install datasets
#%pip install transformers pillow
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from transformers import AutoProcessor, AutoModel
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm # Per la barra di avanzamento
import os

In [9]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id="flaviagiammarino/pubmed-clip-vit-base-patch32"
processor=AutoProcessor.from_pretrained(model_id)
model=AutoModel.from_pretrained(model_id).to(device)
model.eval() #è già pre-addestrato

print("Scaricamento del dataset di radiografie in corso...")
dataset = load_dataset("g-ronimo/NIH-Chest-X-ray-dataset_10k", split="train[:500]")



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Scaricamento del dataset di radiografie in corso...


README.md:   0%|          | 0.00/830 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/457M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/153M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [11]:
dataset['labels']

Column([[0], [0], [11], [4], [11, 13], ...])

In [12]:
dataset['image']

Column([<PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7F6E9FD01010>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7F6EA005BED0>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7F6E91418B90>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7F6E9141CE90>, <PIL.PngImagePlugin.PngImageFile image mode=RGB size=300x300 at 0x7F6E9141D0F0>, ...])

In [ ]:
all_embeddings = []

print("Estrazione embedding visivi in corso....")

with torch.no_grad():
    for item in tqdm(dataset):
        image=item['image']
        
        if image.mode!="RGB":
            image=image.convert("RBG")

        inputs=processor(images=image,return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        vision_outputs=model.get_image_features(**inputs)
        vision_tensor = vision_outputs.pooler_output
        vision_embeddings=F.normalize(vision_tensor,p=2,dim=1)
        all_embeddings.append(vision_embeddings.cpu())


dataset_tensor=torch.cat(all_embeddings,dim=0)
torch.save(dataset_tensor, "nih_chest_embeddings.pt")
print(f"\nDataset salvato. Shape: {dataset_tensor.shape}")

Estrazione embedding visivi in corso....


100%|██████████| 500/500 [00:43<00:00, 11.59it/s]


Fatto! Dataset vettoriale salvato. Shape: torch.Size([500, 512])
